# NorgesGruppen Object Detection — Trening på Google Colab

**Kjør cellene en etter en ved å trykke ▶ til venstre for hver celle.**

Første gang: Gå til `Runtime → Change runtime type → T4 GPU` og trykk Save.

In [ ]:
# Steg 1: Sjekk at du har GPU
!nvidia-smi

In [ ]:
# Steg 2: Installer nødvendige pakker
!pip install ultralytics -q

In [ ]:
# Steg 3: Koble til Google Drive og hent treningsdataene
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# Kopier zip-fil fra Drive til Colab
drive_path = '/content/drive/MyDrive/NM_NGD_coco_dataset.zip'
shutil.copy(drive_path, 'NM_NGD_coco_dataset.zip')
print('NM_NGD_coco_dataset.zip er klar!')

In [ ]:
# Steg 4: Pakk ut zip-filen
import zipfile, os

zip_name = 'NM_NGD_coco_dataset.zip'
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('data/coco')

# Finn annotations.json
for root, dirs, files_list in os.walk('data/coco'):
    for f in files_list:
        if f == 'annotations.json':
            print('Fant annotations.json:', os.path.join(root, f))
        if f.endswith('.jpg'):
            print(f'Fant bilder i: {root}')
            break
    break

In [ ]:
# Steg 5: Konverter COCO-format til YOLO-format
import json
import shutil
from pathlib import Path
from collections import defaultdict

# Finn riktig sti til annotations.json
ann_path = None
for root, dirs, files_list in os.walk('data/coco'):
    for f in files_list:
        if f == 'annotations.json':
            ann_path = Path(root) / f
            img_dir = Path(root).parent / 'images' if (Path(root).parent / 'images').exists() else Path(root) / 'images'
            # also check sibling 'images' folder
            if (Path(root) / 'images').exists():
                img_dir = Path(root) / 'images'
            elif (Path(root).parent / 'images').exists():
                img_dir = Path(root).parent / 'images'
            break

print(f'Annotations: {ann_path}')
print(f'Images dir: {img_dir}')

with open(ann_path) as f:
    coco = json.load(f)

print(f"Antall bilder: {len(coco['images'])}")
print(f"Antall kategorier: {len(coco['categories'])}")
print(f"Antall annotasjoner: {len(coco['annotations'])}")

In [ ]:
# Fortsettelse av konvertering
images = {img['id']: img for img in coco['images']}
cat_id_to_idx = {cat['id']: i for i, cat in enumerate(coco['categories'])}
names = [cat['name'] for cat in coco['categories']]

img_anns = defaultdict(list)
for ann in coco['annotations']:
    if not ann.get('iscrowd', 0):
        img_anns[ann['image_id']].append(ann)

# Lag mapper
yolo_dir = Path('data/yolo')
for split in ('train', 'val'):
    (yolo_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
    (yolo_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

all_ids = list(images.keys())
val_ids = set(all_ids[-25:])   # siste 25 bilder til validering
train_ids = set(all_ids[:-25])

def write_split(ids, split_name):
    for img_id in ids:
        info = images[img_id]
        src = img_dir / info['file_name']
        if not src.exists():
            continue
        W, H = info['width'], info['height']
        dst_img = yolo_dir / 'images' / split_name / info['file_name']
        dst_lbl = yolo_dir / 'labels' / split_name / (src.stem + '.txt')
        shutil.copy2(src, dst_img)
        lines = []
        for ann in img_anns.get(img_id, []):
            x, y, w, h = ann['bbox']
            cx = max(0, min(1, (x + w/2) / W))
            cy = max(0, min(1, (y + h/2) / H))
            nw = max(0, min(1, w / W))
            nh = max(0, min(1, h / H))
            lines.append(f"{cat_id_to_idx[ann['category_id']]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        dst_lbl.write_text('\n'.join(lines))
    print(f'  {split_name}: {len(ids)} bilder')

write_split(train_ids, 'train')
write_split(val_ids, 'val')
print('Konvertering ferdig!')

In [ ]:
# Steg 6: Lag dataset.yaml fil
import yaml

cfg = {
    'path': str(yolo_dir.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(names),
    'names': names
}
yaml_path = yolo_dir / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True)
print(f'Lagret: {yaml_path}')
print(f'Antall klasser: {len(names)}')

In [ ]:
# Steg 7: TREN MODELLEN!
# Dette tar 1-3 timer med GPU.
# YOLOv8s er rask og god nok for en sterk innlevering.

from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # starter med ferdigtrente vekter

model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=1280,
    batch=8,
    workers=2,
    project='runs',
    name='norgesgruppen',
    save=True,
)
print('Trening ferdig!')

In [ ]:
# Steg 8: Last ned best.pt til din PC
from google.colab import files
import shutil
from pathlib import Path

# Finn best.pt
best = Path('runs/norgesgruppen/weights/best.pt')
if not best.exists():
    # Søk etter den
    for p in Path('runs').rglob('best.pt'):
        best = p
        break

print(f'Laster ned: {best}')
files.download(str(best))
print('FERDIG! best.pt er lastet ned til din PC.')

## Etter nedlasting

Når du har `best.pt` på din PC:

1. Gå til Claude Code-terminalen
2. Skriv: `zip -j submission.zip run.py requirements.txt best.pt`
3. Last opp `submission.zip` på competition-siden

Eller si til Claude: **"Jeg har best.pt, hva gjør jeg nå?"**